In [7]:
import pandas as pd
import unicodedata
import re
from pathlib import Path

base_data_dir = Path("./data")
input_files = sorted(base_data_dir.glob("*/Evaluación-Table 1.csv"))
output_file = base_data_dir / "legal-analysis.csv"

def norm(s):
    s = "" if s is None else str(s)
    s = "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c))
    s = re.sub(r"\s+", " ", s).strip().lower()
    return s

rename_map = {
    "Código": "action_id",
    "Acción / Sub-criterios": "action_name",
    "Puntaje\nTotal": "final_score",
    "Puntaje\nLegal\n(40%)": "legal_score_40",
    "Titularidad\ncompetencia\n60%": "legal_ownership_60",
    "Restricciones\njurídicas\n30%": "legal_restrictions_30",
    "Alineación\npolítica\nnacional\n10%": "national_policy_alignment_10",
    "Justificación Legal": "legal_justification",
    "Puntaje\nGob.\n(30%)": "governance_score_30",
    "Capacidad\ntécnica e\ninstitucional\n35%": "technical_institutional_capacity_35",
    "Dependencia\nactores\nexternos\n30%": "external_dependency_30",
    "Instrumento\nlocal\nhabilitante\n25%": "local_enabling_instrument_25",
    "Precedente\nen otras\ncomunas\n10%": "precedent_other_municipalities_10",
    "Justificación Gobernanza": "governance_justification",
    "Puntaje\nFin.\n(30%)": "financing_score_30",
    "Disponibilidad\nfuente\nidentificada\n75%": "funding_source_availability_75",
    "Accesibilidad\ndel\nfinanciamiento\n25%": "funding_accessibility_25",
    "Justificación Financiamiento": "financing_justification",
    "1 Norma / Fuente  →  click para abrir": "legal_reference_1",
    "2 Norma / Fuente  →  click para abrir": "legal_reference_2",
    "3 Norma / Fuente  →  click para abrir": "legal_reference_3",
    "4 Norma / Fuente  →  click para abrir": "legal_reference_4",
    "5 Norma / Fuente  →  click para abrir": "legal_reference_5",
    "6 Norma / Fuente  →  click para abrir": "legal_reference_6",
}

score_cols = [
    "final_score","legal_score_40","legal_ownership_60","legal_restrictions_30",
    "national_policy_alignment_10","governance_score_30","technical_institutional_capacity_35",
    "external_dependency_30","local_enabling_instrument_25","precedent_other_municipalities_10",
    "financing_score_30","funding_source_availability_75","funding_accessibility_25"
]

all_dfs = []

for input_csv in input_files:
    raw = pd.read_csv(input_csv, header=None, dtype=str, keep_default_na=False)
    header_idx = next((i for i in range(len(raw)) if norm(raw.iloc[i, 0]) == "codigo"), None)
    if header_idx is None:
        continue

    header = raw.iloc[header_idx].tolist()
    df = raw.iloc[header_idx + 1:].copy()
    df.columns = header
    df = df.loc[:, [c for c in df.columns if norm(c) != ""]]
    if "Código" in df.columns:
        df = df[df["Código"].astype(str).str.strip().ne("PESOS →")]

    df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

    folder = input_csv.parent.name.lower()
    if "transporte" in folder:
        sector = "transport"
    elif "residuos" in folder:
        sector = "waste"
    elif "energia_tramo1" in folder:
        sector = "energy_tramo1"
    elif "energia_tramo2" in folder:
        sector = "energy_tramo2"
    else:
        sector = folder

    df["sector"] = sector
    df["source_folder"] = input_csv.parent.name
    df["source_file"] = input_csv.name

    for c in score_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    all_dfs.append(df)

legal_analysis = pd.concat(all_dfs, ignore_index=True)
legal_analysis.insert(0, "id", range(1, len(legal_analysis) + 1))
legal_analysis.to_csv(output_file, index=False, encoding="utf-8")

print(f"Saved {output_file} with shape {legal_analysis.shape}")

Saved data/legal-analysis.csv with shape (49, 28)


In [13]:
path = "data/legal-analysis.csv"
df = pd.read_csv(path)

# 1) Rename score columns -> *_numeric
df = df.rename(columns={
    "legal_ownership_60": "legal_ownership_numeric",
    "legal_restrictions_30": "legal_restrictions_numeric",
})

# 2) Add meaning columns based on methodology (0 / 50 / 100)
ownership_map = {
    100: "Municipality has explicit legal authority to act directly.",
     50: "Authority exists but is conditional, ambiguous, or mediated by an enabling instrument.",
      0: "Authority belongs to another level of government; municipality cannot act alone."
}

restrictions_map = {
    100: "No legal restrictions; no additional authorization required.",
     50: "Moderate legal risk; may require prior authorization or face potential legal challenge.",
      0: "There is a legal prohibition/restriction, or legal reform is needed."
}

# Convert safely to numeric first
df["legal_ownership_numeric"] = pd.to_numeric(df["legal_ownership_numeric"], errors="coerce")
df["legal_restrictions_numeric"] = pd.to_numeric(df["legal_restrictions_numeric"], errors="coerce")

# Map to strings
df["legal_ownership_string"] = df["legal_ownership_numeric"].round().map(ownership_map)
df["legal_restrictions_string"] = df["legal_restrictions_numeric"].round().map(restrictions_map)

# Optional: if value is not exactly 0/50/100
df["legal_ownership_string"] = df["legal_ownership_string"].fillna("Non-standard score value.")
df["legal_restrictions_string"] = df["legal_restrictions_string"].fillna("Non-standard score value.")

# View result
df[[
    "action_id",
    "action_name",
    "legal_ownership_numeric",
    "legal_ownership_string",
    "legal_restrictions_numeric",
    "legal_restrictions_string",
    "legal_justification"
]].head()

,action_id,action_name,legal_ownership_numeric,legal_ownership_string,legal_restrictions_numeric,legal_restrictions_string,legal_justification
0,c40_0010,Introducir estándares de eficiencia energética...,50,"Authority exists but is conditional, ambiguous...",50,Moderate legal risk; may require prior authori...,La Ley General de Urbanismo y Construcciones (...
1,c40_0012,Introducir estándares de eficiencia energética...,100,Municipality has explicit legal authority to a...,100,No legal restrictions; no additional authoriza...,La Ley 21.305 (Eficiencia Energética) art. 10°...
2,c40_0015,Rehabilitar edificios residenciales para mejor...,0,Authority belongs to another level of governme...,50,Moderate legal risk; may require prior authori...,La LOCM art. 4° letra f) habilita al municipio...
3,c40_0017,Reacondicionar edificios municipales para mejo...,100,Municipality has explicit legal authority to a...,100,No legal restrictions; no additional authoriza...,La Ley 18.695 art. 5° letra c) habilita al mun...
4,icare_0010,Promover certificaciones de edificios neto-cer...,50,"Authority exists but is conditional, ambiguous...",100,No legal restrictions; no additional authoriza...,La Ley 21.305 y el DS 1/2020 instituyen la Cal...
